In [1]:
!pip install transformers datasets accelerate -q

import torch, math
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed
)
from datasets import Dataset

set_seed(42)

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained('distilgpt2')
model = GPT2LMHeadModel.from_pretrained('distilgpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [3]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)


review_prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

print("=== BASELINE REVIEWS ===\n")
baseline = {}

for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"Prompt: {p}\nOutput: {baseline[p]}\n")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASELINE REVIEWS ===

Prompt: This product is
Output: This product is not designed for a specific purpose.



























































































Prompt: I bought this phone and
Output: I bought this phone and I can't wait to get it done!"

"I just want to apologize, but I still don't know if I'll stop paying for the phone."
"Thank you for it! Do you have any of the phone you get with me? Can't I be honest and tell you my phone got lost in the woods?"
I finally went to the phone, and my phone was ringing.
"Oh wait! I got it! I got the phone

Prompt: The quality of this item
Output: The quality of this item will depend on your current level of use.

























































































In [4]:
corpus = [
    "the old train slowly crossed the bridge while the river shimmered below.",
    "a small cat was sleeping peacefully on the warm window sill.",
    "he forgot his umbrella and had to walk home in the heavy rain.",
    "the library was silent except for the sound of pages turning.",
    "a group of kids were flying colorful kites in the open field.",
    "the smell of fresh bread filled the bakery early in the morning.",
    "she wrote her ideas in a notebook while drinking hot coffee.",
    "the mountain trail was long but the view from the top was beautiful.",
    "birds gathered on the power lines just before sunset.",
    "a cyclist rode quickly through the quiet village road.",
    "the clock on the wall ticked loudly in the empty room.",
    "he planted a tree in the garden hoping it would grow tall someday.",
    "the night sky was full of bright stars and a cool breeze.",
    "someone left a red backpack on the park bench.",
    "the dog happily chased a ball across the grassy field.",
    "a street musician played guitar near the busy market.",
    "the bus arrived late and everyone looked a little tired.",
    "soft music played in the background of the small cafe.",
    "the fisherman waited patiently by the calm lake.",
    "a paper airplane glided slowly across the classroom.",

    "a bright balloon floated away into the blue sky.",
    "the chef carefully prepared the meal in the quiet kitchen.",
    "a child laughed loudly while running through the park.",
    "snow slowly covered the rooftops during the silent night.",
    "a train whistle echoed across the empty valley.",
    "someone painted a colorful mural on the city wall.",
    "the wind moved gently through the tall grass.",
    "a lantern glowed softly outside the wooden cabin.",
    "students gathered in the hall before the lecture started.",
    "a squirrel quickly climbed up the tall oak tree.",
    "the old bookstore smelled like paper and history.",
    "a boat drifted slowly across the calm river.",
    "people waited patiently at the busy train station.",
    "a candle flickered quietly on the dinner table.",
    "clouds moved slowly across the bright afternoon sky.",
    "the gardener watered the flowers in the early morning.",
    "a distant thunder rolled across the dark sky.",
    "the path through the forest was covered with fallen leaves.",
    "children built a sandcastle near the ocean waves.",
    "a quiet breeze passed through the open window."
]

In [5]:

dataset = Dataset.from_dict({"text": corpus})

tokenized = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=128, padding="max_length"),
    batched=True,
    remove_columns=["text"]
)

split = tokenized.train_test_split(test_size=0.15, seed=42)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

In [6]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gpt2-reviews",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.571771
20,3.647208
30,3.012054
40,2.577207
50,2.239682
60,1.820597
70,1.669737
80,1.551637
90,1.494174


TrainOutput(global_step=90, training_loss=2.509341017405192, metrics={'train_runtime': 13.1426, 'train_samples_per_second': 25.87, 'train_steps_per_second': 6.848, 'total_flos': 11105111900160.0, 'train_loss': 2.509341017405192, 'epoch': 10.0})

In [7]:
eval_res = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_res['eval_loss']):.2f}\n")

print("=== FINE-TUNED REVIEWS ===\n")

for p in review_prompts:
    ft = generate_text(model, tokenizer, p)
    print(f"Prompt: {p}")
    print(f"Baseline:   {baseline[p][:120]}")
    print(f"Fine-Tuned: {ft[:120]}\n")


Perplexity: 41.74

=== FINE-TUNED REVIEWS ===

Prompt: This product is
Baseline:   This product is not designed for a specific purpose.




































































Fine-Tuned: This product is a quiet portable music player that lasts only 1 minute. The sound of music fills the room quickly and yo

Prompt: I bought this phone and
Baseline:   I bought this phone and I can't wait to get it done!"

"I just want to apologize, but I still don't know if I'll stop pa
Fine-Tuned: I bought this phone and thought it was going to work. All the lights turned on and the sound of footsteps echoed through

Prompt: The quality of this item
Baseline:   The quality of this item will depend on your current level of use.






















































Fine-Tuned: The quality of this item has improved significantly over the past few days. I would recommend it as a gift exchange star



In [8]:
tokenizer2 = GPT2Tokenizer.from_pretrained('distilgpt2')
model2 = GPT2LMHeadModel.from_pretrained('distilgpt2')

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
recipe_prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare a chocolate cake"
]

baseline2 = {}

print("=== BASELINE RECIPES ===\n")

for p in recipe_prompts:
    baseline2[p] = generate_text(model2, tokenizer2, p)
    print(f"Prompt: {p}\nOutput: {baseline2[p]}\n")

=== BASELINE RECIPES ===

Prompt: To make butter chicken
Output: To make butter chicken.

This post may contain links to Amazon or other partners; your purchases via these links can benefit Serious Eats. Read more about our affiliate linking policy.

Prompt: For pasta carbonara
Output: For pasta carbonara pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta

Prompt: To prepare a chocolate cake
Output: To prepare a chocolate cake. For a cookie, go to the baking sheet, place

In [11]:
recipes = [
    "start by washing the rice and soaking it in water for twenty minutes.",
    "heat oil in a pan and add cumin seeds until they begin to crackle.",
    "add chopped onions and cook until they become soft and light brown.",
    "mix in tomatoes and spices and cook until everything blends well.",
    "add the rice with water and let it simmer until fully cooked.",

    "to make vegetable soup boil water in a large pot.",
    "add chopped carrots beans and potatoes.",
    "season with salt pepper and herbs.",
    "let the vegetables cook until tender.",
    "serve the soup warm with bread.",

    "for a simple sandwich toast two slices of bread.",
    "spread butter or mayonnaise on each slice.",
    "add lettuce tomato and cucumber slices.",
    "place cheese between the bread slices.",
    "cut the sandwich in half and serve."
]

dataset2 = Dataset.from_dict({"text": recipes})

tok2 = dataset2.map(
    lambda x: tokenizer2(x["text"], truncation=True, max_length=128, padding="max_length"),
    batched=True,
    remove_columns=["text"]
)

split2 = tok2.train_test_split(test_size=0.15, seed=42)

Map:   0%|          | 0/15 [00:00<?, ? examples/s]

In [12]:
collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

args2 = TrainingArguments(
    output_dir="./gpt2-recipes",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=split2["train"],
    eval_dataset=split2["test"],
    data_collator=collator2
)

trainer2.train()

Step,Training Loss
10,3.867692
20,2.545374
30,2.037947


TrainOutput(global_step=30, training_loss=2.8170042673746747, metrics={'train_runtime': 2.4788, 'train_samples_per_second': 48.411, 'train_steps_per_second': 12.103, 'total_flos': 3919451258880.0, 'train_loss': 2.8170042673746747, 'epoch': 10.0})

In [13]:
eval2 = trainer2.evaluate()
print(f"Perplexity: {math.exp(eval2['eval_loss']):.2f}\n")

print("=== FINE-TUNED RECIPES ===\n")

for p in recipe_prompts:
    ft = generate_text(model2, tokenizer2, p)
    print(f"Prompt: {p}")
    print(f"Baseline:   {baseline2[p][:120]}")
    print(f"Fine-Tuned: {ft[:120]}\n")

Perplexity: 114.80

=== FINE-TUNED RECIPES ===

Prompt: To make butter chicken
Baseline:   To make butter chicken.

This post may contain links to Amazon or other partners; your purchases via these links can ben
Fine-Tuned: To make butter chicken slices in ½ cup water and combine until they become soft and soft.
































Prompt: For pasta carbonara
Baseline:   For pasta carbonara pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta pasta past
Fine-Tuned: For pasta carbonara chips with onion garlic and cook until tender.























































Prompt: To prepare a chocolate cake
Baseline:   To prepare a chocolate cake. For a cookie, go to the baking sheet, place a small cookie sheet on top, and place the cook
Fine-Tuned: To prepare a chocolate cake pan and bake until lightly golden brown.























































In [14]:
import pandas as pd

results_reviews = []

for p in review_prompts:
    ft = generate_text(model, tokenizer, p)

    results_reviews.append({
        "Prompt": p,
        "Baseline": baseline[p],
        "Fine_Tuned": ft
    })

df_reviews = pd.DataFrame(results_reviews)

# Save as CSV
df_reviews.to_csv("product_review_results.csv", index=False)

print("Saved product review results!")
df_reviews

Saved product review results!


,Prompt,Baseline,Fine_Tuned
0,This product is,This product is not designed for a specific pu...,This product is currently in the download gall...
1,I bought this phone and,I bought this phone and I can't wait to get it...,I bought this phone and it sounded cool. I not...
2,The quality of this item,The quality of this item will depend on your c...,The quality of this item has improved signific...


In [15]:
results_recipes = []

for p in recipe_prompts:
    ft = generate_text(model2, tokenizer2, p)

    results_recipes.append({
        "Prompt": p,
        "Baseline": baseline2[p],
        "Fine_Tuned": ft
    })

df_recipes = pd.DataFrame(results_recipes)

df_recipes.to_csv("recipe_results.csv", index=False)

print("Saved recipe results!")
df_recipes

Saved recipe results!


,Prompt,Baseline,Fine_Tuned
0,To make butter chicken,To make butter chicken.\n\nThis post may conta...,To make butter chicken slices into small piece...
1,For pasta carbonara,For pasta carbonara pasta pasta pasta pasta pa...,For pasta carbonara bread slices bread slices ...
2,To prepare a chocolate cake,"To prepare a chocolate cake. For a cookie, go ...",To prepare a chocolate cake pan until the butt...
